# Milestone 2 — Encoder Comparison: CLIP · BERT · GPT-2

*Optional extension: autoregressive text encoders.*

No generation and no pipeline. The same curated captions go through three text encoders, and the
question is which of them represents **meaning** rather than surface word overlap.

**The measurement.** The two captions of one image describe the same scene, so an encoder that
represents meaning should place them closer together than two unrelated captions. The gap between
those two cosines is the score. The failure mode to watch for is a large sibling cosine alongside an
equally large random cosine: that means everything has been mapped into one narrow cone, so the
ranking carries almost no information.

**The point.** It is tempting to say CLIP beats GPT-2 because GPT-2 is causally masked. That is
wrong — CLIP's text tower is causally masked too. What separates them is the training objective:
contrastive image-text matching against next-token prediction.

Each encoder's functions are named apart (`encode_clip_text`, `encode_bert_text`,
`encode_gpt2_text`) because all three would otherwise be called `encode_text`.

---
## 1. Setup

Everything written goes to `/kaggle/working/outputs/`, including the GPT-2 vectors this
notebook produces. The captions are read straight from the attached Dataset, which is never written
to. Off Kaggle both fall back to `./data/processed/` and `./outputs/`.

In [ ]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import transformers
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()


# Returns (processed data directory, output directory). The Dataset is read straight
# from /kaggle/input -- nothing writes under it, so copying 390 MB into the working
# directory first would buy nothing.
def resolve_paths():
    if Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir():
        found = sorted(Path("/kaggle/input").glob("**/processed/pairs.parquet"))
        if not found:
            raise RuntimeError(
                "Attach the Dataset holding processed/pairs.parquet. Under /kaggle/input: "
                + (", ".join(p.name for p in Path("/kaggle/input").iterdir()) or "nothing"))
        return found[0].parent, Path("/kaggle/working/outputs")
    return Path("data/processed"), Path("outputs")


PROCESSED, OUT_DIR = resolve_paths()
PLOTS_DIR, TABLES_DIR = OUT_DIR / "plots", OUT_DIR / "tables"
# The GPT-2 vectors are an output of this notebook, not source data, so they go to
# the output directory -- which is also why the Dataset never needs to be writable.
EMB_DIR = OUT_DIR / "embeddings"
for d in (PLOTS_DIR, TABLES_DIR, EMB_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
BATCH_SIZE = 64
CLIP_MODEL = "openai/clip-vit-base-patch32"
BERT_MODEL = "google-bert/bert-base-uncased"
GPT2_MODEL = "openai-community/gpt2"
CLIP_MAX_TOKENS, TEXT_MAX_TOKENS = 77, 64

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"torch {torch.__version__} | transformers {transformers.__version__}")
print(f"device  : {DEVICE}")
print(f"data    : {PROCESSED}")
print(f"outputs : {OUT_DIR}")

### The captions and their sibling pairs

Each image contributes two captions. `PAIR_A[i]` and `PAIR_B[i]` are the row indices of the two
captions describing image *i*, which is what makes the sibling comparison possible.

In [ ]:
manifest = pd.read_parquet(PROCESSED / "pairs.parquet")
CAPTIONS = manifest["caption"].tolist()

first = manifest[manifest["cap_rank"] == 0].set_index("image_id")["pair_idx"]
second = manifest[manifest["cap_rank"] == 1].set_index("image_id")["pair_idx"]
shared = first.index.intersection(second.index)
PAIR_A, PAIR_B = first.loc[shared].to_numpy(), second.loc[shared].to_numpy()

print(f"captions      : {len(CAPTIONS):,}")
print(f"sibling pairs : {len(PAIR_A):,} images with two captions each\n")
for a, b in zip(PAIR_A[:3], PAIR_B[:3]):
    print(f"  {CAPTIONS[a]}\n  {CAPTIONS[b]}\n")

---
## 2. Is CLIP's text tower really causal?

Asked of the installed library rather than taken on trust. CLIP builds a causal attention mask
inside its text transformer the same way GPT-2 does.

In [ ]:
import inspect
from transformers.models.clip import modeling_clip

source = inspect.getsource(modeling_clip)
for marker in ["_create_4d_causal_attention_mask", "causal_attention_mask", "is_causal"]:
    print(f"  {marker:<36}{'present' if marker in source else 'absent'}")

print("\nBoth mask future tokens. Only the training objective differs.")

---
## 3. CLIP

The pooled 512-d vector from the joint text-image space.

In [ ]:
def pooled(out):
    return out.pooler_output if hasattr(out, "pooler_output") else out


@torch.no_grad()
def encode_clip_text(texts, tokenizer, model, batch_size=BATCH_SIZE):
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="CLIP"):
        toks = tokenizer(texts[i:i + batch_size], padding="max_length",
                         max_length=CLIP_MAX_TOKENS, truncation=True,
                         return_tensors="pt").to(DEVICE)
        out.append(pooled(model.get_text_features(**toks)).float().cpu())
    return torch.cat(out).numpy()


timings = {}
clip_tokenizer = CLIPTokenizer.from_pretrained(CLIP_MODEL)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE, torch.float32).eval()

t0 = time.time()
clip_joint = encode_clip_text(CAPTIONS, clip_tokenizer, clip_model)
timings["CLIP ViT-B/32 (pooled)"] = time.time() - t0

print(f"\n{clip_joint.shape}  {clip_joint.dtype}  in {timings['CLIP ViT-B/32 (pooled)']:.1f}s")
del clip_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

---
## 4. BERT

`bert-base-uncased` is the same architecture class and roughly the same size as CLIP's text tower,
but trained with masked language modelling on text alone. It has never seen an image. That
difference is the point: both produce 768-d sentence vectors that look equally reasonable on paper,
so the only way to tell them apart is to measure them.

BERT ships a `pooler_output`, but it is the tanh-projected `[CLS]` state trained for next-sentence
prediction, not for sentence similarity — using it as a sentence embedding is a well-documented
trap. Two honest alternatives are computed instead.

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    return (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


@torch.no_grad()
def encode_bert_text(texts, tokenizer, model, batch_size=BATCH_SIZE):
    cls_out, mean_out = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="BERT"):
        toks = tokenizer(texts[i:i + batch_size], padding="max_length",
                         max_length=TEXT_MAX_TOKENS, truncation=True,
                         return_tensors="pt").to(DEVICE)
        hidden = model(**toks).last_hidden_state
        cls_out.append(hidden[:, 0].float().cpu())
        mean_out.append(mean_pool(hidden, toks["attention_mask"]).float().cpu())
    return torch.cat(cls_out).numpy(), torch.cat(mean_out).numpy()


bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(DEVICE, torch.float32).eval()

t0 = time.time()
bert_cls, bert_mean = encode_bert_text(CAPTIONS, bert_tokenizer, bert_model)
timings["BERT (CLS / mean)"] = time.time() - t0

print(f"\n{bert_cls.shape} CLS, {bert_mean.shape} mean, in {timings['BERT (CLS / mean)']:.1f}s")
del bert_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

---
## 5. GPT-2

Pooling needs care for a causal model. Only the **last** token has attended to the whole sentence,
so it is the natural sentence vector — but GPT-2 pads on the right, so position −1 is usually
padding, not the sentence. `last_token_pool` finds the last real token instead.

Mean pooling is reported alongside it because a causal model's early positions have seen almost
nothing, which drags the mean toward the first few words.

GPT-2 ships no padding token, so the tokenizer is pointed at the EOS token. Every padded position
is masked out of both poolings, so this costs nothing.

In [ ]:
def last_token_pool(last_hidden_state, attention_mask):
    lengths = attention_mask.sum(dim=1)
    index = (lengths - 1).clamp(min=0).to(last_hidden_state.device)
    rows = torch.arange(len(last_hidden_state), device=last_hidden_state.device)
    return last_hidden_state[rows, index]


@torch.no_grad()
def encode_gpt2_text(texts, tokenizer, model, batch_size=BATCH_SIZE):
    last_out, mean_out = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc="GPT-2"):
        toks = tokenizer(texts[i:i + batch_size], padding="max_length",
                         max_length=TEXT_MAX_TOKENS, truncation=True,
                         return_tensors="pt").to(DEVICE)
        hidden = model(**toks).last_hidden_state
        last_out.append(last_token_pool(hidden, toks["attention_mask"]).float().cpu())
        mean_out.append(mean_pool(hidden, toks["attention_mask"]).float().cpu())
    return torch.cat(last_out).numpy(), torch.cat(mean_out).numpy()


gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL)
if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = AutoModel.from_pretrained(GPT2_MODEL).to(DEVICE, torch.float32).eval()

print(f"pad token set to {gpt2_tokenizer.pad_token!r}")
t0 = time.time()
gpt2_last, gpt2_mean = encode_gpt2_text(CAPTIONS, gpt2_tokenizer, gpt2_model)
timings["GPT-2 (last / mean)"] = time.time() - t0

print(f"\n{gpt2_last.shape} last-token, {gpt2_mean.shape} mean, in {timings['GPT-2 (last / mean)']:.1f}s")
np.save(EMB_DIR / "gpt2_last.npy", gpt2_last)
np.save(EMB_DIR / "gpt2_mean.npy", gpt2_mean)
print(f"saved to {EMB_DIR}")

del gpt2_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

---
## 6. Sibling agreement

Every encoder gets the same pairs and the same shuffle, so the only variable is the encoder.

In [ ]:
def sibling_gap(encoders, pair_a, pair_b, seed=SEED):
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(len(pair_b))
    scores = {}
    for label, emb in encoders.items():
        v = F.normalize(torch.from_numpy(emb.astype(np.float32)), dim=-1)
        sib = (v[pair_a] * v[pair_b]).sum(-1).numpy()
        rnd = (v[pair_a] * v[pair_b][shuffled]).sum(-1).numpy()
        scores[label] = {"sibling_cos": float(sib.mean()),
                         "random_cos": float(rnd.mean()),
                         "gap": float(sib.mean() - rnd.mean())}
    return scores


ENCODERS = {
    "CLIP ViT-B/32 (pooled)": clip_joint,
    "BERT [CLS]": bert_cls,
    "BERT mean": bert_mean,
    "GPT-2 last token": gpt2_last,
    "GPT-2 mean": gpt2_mean,
}

scores = sibling_gap(ENCODERS, PAIR_A, PAIR_B)
table = (pd.DataFrame(scores).T
         .rename(columns={"sibling_cos": "sibling", "random_cos": "random"})
         .sort_values("gap", ascending=False))
table["ratio"] = (table["sibling"] / table["random"]).round(3)
display(table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
y = np.arange(len(table))
ax.barh(y - 0.2, table["sibling"], height=0.38, label="two captions of the same image")
ax.barh(y + 0.2, table["random"], height=0.38, label="two unrelated captions")
ax.set_yticks(y); ax.set_yticklabels(table.index); ax.invert_yaxis()
ax.set_xlabel("mean cosine similarity")
ax.set_title("Sibling agreement: does the encoder represent meaning?")
ax.legend(); ax.grid(axis="x", alpha=0.3)
for i, gap in enumerate(table["gap"]):
    ax.annotate(f"gap {gap:.3f}",
                (max(table["sibling"].iloc[i], table["random"].iloc[i]) + 0.01, i),
                va="center", fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "m2_encoder_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Reading the numbers

Read the **gap**, not the sibling cosine. A row with a high sibling cosine *and* a high random
cosine has collapsed its captions into a narrow cone — every sentence looks like every other, and
the ordering is nearly meaningless. The `ratio` column makes that visible: how many times more
similar a true sibling is than a random pair.

For GPT-2, compare its two rows against each other. Last-token pooling is the theoretically right
choice for a causal model, while mean pooling averages in early positions that have seen almost
nothing. Which actually wins here is the finding, not the assumption.

In [ ]:
best = table.index[0]
summary = {
    "n_captions": len(CAPTIONS),
    "n_sibling_pairs": int(len(PAIR_A)),
    "encode_seconds": {k: round(v, 1) for k, v in timings.items()},
    "scores": {k: {m: round(float(x), 4) for m, x in v.items()} for k, v in scores.items()},
    "ranking_by_gap": table.index.tolist(),
    "best": best,
    "note": ("CLIP's text tower is causally masked too; the training objective, not the mask, "
             "is what separates it from GPT-2."),
}
(TABLES_DIR / "milestone_2_encoders.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(TABLES_DIR / "milestone_2_encoders.json")
print(f"\nbest separation: {best}  (gap {table.loc[best, 'gap']:.4f}, "
      f"{table.loc[best, 'ratio']:.2f}x random)")